# **Reproducción de audios**

## **Librerías y módulos necesarios**

In [1]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import Audio, display
import os

## **Datos**

In [2]:
audio_dir = '../data/train_audio'
csv_filename = '../data/observaciones_top20_especies.csv'
FILENAME_COLUMN = 'filename' 

try:
    df = pd.read_csv(csv_filename)
except FileNotFoundError:
    print(f"ERROR: El archivo '{csv_filename}' no se encuentra.")
    df = pd.DataFrame()

In [3]:
if not df.empty and FILENAME_COLUMN in df.columns:
    
    df_filtered = df.copy() 
    df_filtered['combined_label'] = (
        df_filtered['primary_label'].astype(str) + 
        ' (' + df_filtered['scientific_name'].astype(str) + ')'
    )
    
    species_options = sorted(df_filtered['combined_label'].unique().tolist())
    
    if species_options:
        initial_species = species_options[0]
        initial_audios = sorted(df_filtered[
            df_filtered['combined_label'] == initial_species
        ][FILENAME_COLUMN].unique().tolist()) 
    else:
        initial_species = "No data"
        initial_audios = ["No hay audios disponibles"]
    
else:
    print("ERROR: No se pudieron preparar los datos.")
    species_options = ['No data']
    initial_species = "No data"
    initial_audios = ["No hay audios disponibles"]

## **Funciones**

In [4]:
species_dropdown = widgets.Dropdown(
    options=species_options,
    value=initial_species if species_options else None,
    description='Seleccionar Especie:',
    disabled=not bool(species_options),
    style={'description_width': 'initial'}
)

audio_dropdown = widgets.Dropdown(
    options=initial_audios,
    value=initial_audios[0] if initial_audios and initial_audios[0] != "No hay audios disponibles" else None,
    description='Seleccionar Audio:',
    disabled=not bool(initial_audios) or initial_audios[0] == "No hay audios disponibles",
    style={'description_width': 'initial'}
)

audio_output = widgets.Output()

In [5]:
class MockChange:
    def __init__(self, new_value):
        self.new = new_value

def display_audio_player(change):
    """Muestra el reproductor de audio."""
    selected_filename = change.new
    
    with audio_output:
        audio_output.clear_output() 
        
        if selected_filename and selected_filename != "No hay audios disponibles":
            # La ruta se construye usando el nombre completo de 'filename'
            audio_path = os.path.join(audio_dir, selected_filename)
            print(f"Reproduciendo: {selected_filename}")
            
            try:
                display(Audio(audio_path))
            except PermissionError:
                 print("\n ERROR DE PERMISO: Verifica que el archivo no esté siendo usado por otro programa.")
            except FileNotFoundError:
                 print(f"\n ERROR DE ARCHIVO: Archivo no encontrado en la ruta: {audio_path}")

        else:
             print("Selecciona una especie y un audio para reproducir.")


def update_audio_options(change):
    selected_species_label = change.new
    
    with audio_output:
        audio_output.clear_output()
        
    if selected_species_label and selected_species_label != 'No data':
        
        new_audios = sorted(df_filtered[
            df_filtered['combined_label'] == selected_species_label
        ][FILENAME_COLUMN].unique().tolist())
        
        audio_dropdown.options = new_audios
        audio_dropdown.value = new_audios[0] if new_audios else None
        audio_dropdown.disabled = not bool(new_audios)
    else:
        audio_dropdown.options = ["No hay audios disponibles"]
        audio_dropdown.value = None
        audio_dropdown.disabled = True
    
    if audio_dropdown.value:
         mock_event_object = MockChange(audio_dropdown.value)
         display_audio_player(mock_event_object)
    else:
        with audio_output:
            audio_output.clear_output()

species_dropdown.observe(update_audio_options, names='value')
audio_dropdown.observe(display_audio_player, names='value')

In [6]:
print("\n\n REPRODUCCIÓN DE AUDIOS")
print("1. Selecciona una especie (código - nombre científico).")
print("2. Selecciona un audio para reproducirlo.")
print("-" * 50)

interface_layout = widgets.VBox([
    species_dropdown,
    audio_dropdown
])


display(interface_layout, audio_output)

if audio_dropdown.value:
    mock_event_object = MockChange(audio_dropdown.value)
    display_audio_player(mock_event_object)



 REPRODUCCIÓN DE AUDIOS
1. Selecciona una especie (código - nombre científico).
2. Selecciona un audio para reproducirlo.
--------------------------------------------------


Output()